# Homework 5: Linear algebra

**Course title:** Computer Simulation Methods in Physics (Phys 4071/5071; Fall 2025)  
**This homework assignment is due October 1, 2025, at 8 am.**   
(Submission via the shared `GitHub` repository.)

**Before you submit your homework, make sure that you click `Kernel > Restart & Run All`**.

## LU Decomposition

(a) Find the LU decompositions of the following three matrices (no pivoting necessary):

$$
A_1 = 
\begin{pmatrix}
 2 & 1 \\
 -1 & 2 
\end{pmatrix} 
, \quad 
A_2 = 
\begin{pmatrix}
 2 & -1 & 0 \\
 -1 & 2 & -1 \\
 0 & -1 & 2 \\
 \end{pmatrix} 
, \quad 
A_3 = 
 \begin{pmatrix}
 3 & 0 & 0 \\
 -1 & 2 & 0 \\
 2 & 1 & 2 \\
 \end{pmatrix} \,.
$$

Show that your analytic calculations are correct using appropriate `Python` libraries.

[Question 1 Part A.](https://drive.google.com/file/d/1bfxtSvK6VjfTz7u5xGzG1PvTL4k9LK0-/view?usp=sharing)

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from scipy.linalg import lu 
from sympy import init_printing, Matrix, pprint 


A_1 = np.array([[2,1], [-1,2]], float) 
A_2 = np.array([[2, -1, 0], [-1, 2, -1], [0, -1, 2]], float) 
A_3 = np.array([[3, 0, 0], [-1, 2, 0], [2, 1, 2]], float) 

N_1 = len(A_1[:,1]) 
N_2 = len(A_2[:,1]) 
N_3 = len(A_3[:,1]) 


def divisor(N, Mat):
    '''
    Row-normalize a square matrix by its diagonal entries.
    Function divides each row m by the diagonal element Mat[m, m].
    Operates in-place on Mat. Assumes all diagonal entries are nonzero.

    Parameters
    ----------
    N : int
        Matrix size (rows = cols = N).
    Mat : numpy.ndarray
        Square matrix to be normalized in-place.

    Returns
    -------
    numpy.ndarray
        The same matrix object, after in-place row scaling.
    '''
    for m in range(N):
        div = Mat[m, m]
        Mat[m, :] /= div
    return Mat


def Upper(N, U_Mat):
    '''
    Forward elimination to form an upper-triangular matrix (no pivoting).
    For each pivot column i, eliminates U_Mat[r, i] for r > i via
     R_r - [(U_Mat[r,i]/U_Mat[i,i]) * R_i] -> R_r . 

    Parameters
    ----------
    N : int
        Matrix size (rows = cols = N).
    U_Mat : numpy.ndarray
        Working matrix modified in-place.

    Returns
    -------
    numpy.ndarray
        The same array, now upper-triangular (if pivots are nonzero).

    '''
    for i in range(N - 1):
        for r in range(i + 1, N):
            m = U_Mat[r, i] / U_Mat[i, i]
            U_Mat[r, :] -= m * U_Mat[i, :]
    return U_Mat


def Lower(N, L_Mat):
    '''
    Make a lower-triangular copy and scale rows beneath each pivot; set unit diagonal.
    Creates a float copy B of L_Mat, takes L = tril(B), then for each i,
    divides rows k = i...N by the current diagonal L[i,i], and finally
    sets L[i,i] = 1.0. 

    Parameters
    ----------
    N : int
        Matrix size (rows = cols = N).
    L_Mat : numpy.ndarray
        Input matrix (unchanged).

    Returns
    -------
    numpy.ndarray
        New lower-triangular matrix with unit diagonal as produced by this routine.

    '''
    B = np.array(L_Mat, dtype=float, copy=True)
    L = np.tril(B)
    for i in range(N):
        m = L[i, i]
        for k in range(i, N):
            L[k ,i] /= m
        L[i, i] = 1.0
    return L


def Back_Sub(N, L_Mat):
    '''
    Back-eliminate above the diagonal, then row-normalize to unit diagonal.
    Runs Gauss–Jordan style elimination from bottom to top:
    for each pivot row i, eliminates entries above it using
   R_r - (L_Mat[r,i]/L_Mat[i,i]) * R_i -> R_r  .
    After elimination, calls divisor(N, L_Mat) to set the diagonal to 1.
   

    Parameters
    ----------
    N : int
        Matrix size (rows = cols = N).
    L_Mat : numpy.ndarray
        Working matrix modified in-place.

    Returns
    -------
    numpy.ndarray
        The same matrix after back-elimination and normalization.

    '''
    for i in range(N - 1, -1, -1):
        piv = L_Mat[i, i]
        for r in range(i - 1, -1, -1):
            m = L_Mat[r, i] / piv
            L_Mat[r, :] -= m * L_Mat[i, :]
    L_Mat = divisor(N, L_Mat)
    return L_Mat



print("Matrix A_1 Decompositions\n")
print("L_1 =\n", Lower(N_1, A_1))
print("U_1 =\n", Upper(N_1, A_1), "\n")

print("Matrix A_2 Decompositions\n")
print("L_2 =\n", Lower(N_2, A_2))
print("U_2 =\n", Upper(N_2, A_2), "\n")

print("Matrix A_3 Decompositions\n")
print("L_3 =\n", Lower(N_3, A_3))
print("U_3 =\n", Upper(N_3, A_3))








Matrix A_1 Decompositions

L_1 =
 [[ 1.   0. ]
 [-0.5  1. ]]
U_1 =
 [[2.  1. ]
 [0.  2.5]] 

Matrix A_2 Decompositions

L_2 =
 [[ 1.   0.   0. ]
 [-0.5  1.   0. ]
 [ 0.  -0.5  1. ]]
U_2 =
 [[ 2.         -1.          0.        ]
 [ 0.          1.5        -1.        ]
 [ 0.          0.          1.33333333]] 

Matrix A_3 Decompositions

L_3 =
 [[ 1.          0.          0.        ]
 [-0.33333333  1.          0.        ]
 [ 0.66666667  0.5         1.        ]]
U_3 =
 [[3. 0. 0.]
 [0. 2. 0.]
 [0. 0. 2.]]


(b) Solve the linear equations $A_2 x = b$ for 

$$
b = 
\begin{pmatrix}
1\\2\\3
\end{pmatrix}
$$

using your LU decomposition and back-substitution in `Python`. Do not use any existing `Python` library. Show that your analytic calculations are correct using appropriate `Python` libraries.

[Question 1 Part B.](https://drive.google.com/file/d/1LGYJoZf0HddgC0Z9GevVZfL10qDx05Z9/view?usp=sharing)

In [2]:

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from scipy.linalg import lu 
from sympy import init_printing, Matrix, pprint 
L = np.array([[ 1. ,  0. ,  0. ],
              [-0.5,  1. ,  0. ],
              [ 0. , -0.5,  1. ]], float)
U = np.array([[2., -1., 0.], 
             [0., 3./2., -1./2.], 
             [0., 0., 4./3.]], float)
b = np.array([1, 2, 3], float).reshape(L.shape[0], 1)   




def Forward(L, b):
    '''
    Reduce [L | b] with L lower-triangular to [I | y] by forward elimination.
    Performs row operations on the augmented matrix [L | b] to eliminate entries
    below the diagonal of L, carrying b along. Returns the right-hand side column y.

    Parameters
    ----------
    L : numpy.ndarray
        Square lower-triangular matrix (n x n).
    b : array-like
        Right-hand side. Shape (n,) or (n,1). Will be augmented as a column.

    Returns
    -------
    numpy.ndarray
        y as a 1-D array of length n (solution of L y = b).
    '''
    L = np.array(L, dtype=float, copy=True)
    Lb = np.hstack([L, np.array(b, dtype=float).reshape(L.shape[0], 1)])
    N = L.shape[0]
    for i in range(N):
        for r in range(i + 1, N):
            m = Lb[r, i] / Lb[i, i]
            Lb[r, :] -= m * Lb[i, :]
            Lb[r, i] = 0.0
    y = Lb[:, -1]
    return y


def Back_Sub(U_s, y_s):
    '''
    Reduce [U | y] with U upper-triangular to [I | x] by back substitution.
    Performs row operations on the augmented matrix [U | y] to eliminate entries
    above the diagonal of U (from bottom row upward), then normalizes each row
    by its diagonal using divisor to obtain the identity on the left. Returns x.

    Parameters
    ----------
    U_s : numpy.ndarray
        Square upper-triangular matrix (n x n). 
    y_s : array-like
        Right-hand side. Shape (n,) or (n,1). Will be augmented as a column.

    Returns
    -------
    numpy.ndarray
        x as a 1-D array of length n (solution of U x = y).
    '''
    U = np.array(U_s, dtype=float, copy=True)
    y = np.array(y_s, dtype=float).reshape(U.shape[0], 1)
    Uy = np.hstack([U, y])
    N = U.shape[0]
    for i in range(N - 1, -1, -1):
        piv = Uy[i, i]
        for r in range(i - 1, -1, -1):
            m = Uy[r, i] / piv
            Uy[r, :] -= m * Uy[i, :]
    x = divisor(N, Uy)
    x_final = x[:, -1]
    return x_final




y = Forward(L,b)
print("y_vector and solution x:\n")
print(y)
print(Back_Sub(U,y))


y_vector and solution x:

[1.   2.5  4.25]
[1.86458333 2.72916667 3.1875    ]


## QR Algorithm

(a) Write a `Python` function that takes as its argument a real square matrix $A$ and returns the two matrices $Q$ and $R$ that form its QR decomposition as a test case, try out your function on the matrix

$$
A = 
\begin{pmatrix}
 1 & 4 & 8 & 4 \\
 4 & 2 & 3 & 7 \\
 8 & 3 & 6 & 9 \\
 4 & 7 & 9 & 2 
 \end{pmatrix} \,.
$$

Check the results by multiplying $Q$ and $R$ together to recover the original matrix $A$.
Don't use any existing implementation of the QR decomposition.

Compare the output of your function to `numpy.linalg.qr()`. Is the QR decomposition unique, in general?

In [3]:
#Question 2 Part A.) 
import numpy as np 
import pandas as pd 
import math
import matplotlib.pyplot as plt 
from scipy.linalg import lu 
import numpy.linalg as linalg
from sympy import init_printing, Matrix, pprint 


A = np.array([[1, 4, 8, 4], [4, 2, 3, 7], [8, 3, 6, 9], [4, 7, 9, 2]], float)
def QR_Decomp(B): 
    '''
    A Gram-Schmidt process is applied to the column vector of matrix B input. 
    Main equations used R_ij = Q^T*v; v = v - R_ij*Q; R_jj = 1/sqrt(n^2); and Q_ij = v/R_jj. 
    The final goal is to obtain the R and Q matrices in decomposed form from the input matrix. 
    
    Parameters
    ----------
    B : numpy.ndarray
       Input a real and square matrix. 

    Returns
    -------
    numpy.ndarray
        Both decomposed from B: Q and R matrices are output.
    '''
    m,n = B.shape
    Q = np.zeros((m,n))
    R = np.zeros((n,n))
    for j in range(n): 
        v = B[:,j].copy()
        for i in range(j):
            R[i,j] = np.dot(Q[:, i], v)
            v -= R[i,j]*Q[:, i]
        R[j,j] = linalg.norm(v)
        Q[:,j] = v/(R[j,j]) 
    return Q,R 





print("QR-> matrices by function make\n")
print(QR_Decomp(A))


print("QR-> matrices by python built-function\n")
print(linalg.qr(A))

        
Q_func, R_func = QR_Decomp(A)
print("\n QR-function matrix multiplication \n")
print(np.dot(Q_func, R_func))


Q_python, R_python = linalg.qr(A)
print("\n QR python-made matrix multiplication \n")
print(np.dot(Q_python, R_python))


"""

This QR decomposition are'nt unique because the only sign convention is the main difference in the Gram-Schmidt process
Then the resultant should still give a when you do a matrix product of QR. 

"""






QR-> matrices by function make

(array([[ 0.10153462,  0.558463  ,  0.80981107,  0.1483773 ],
       [ 0.40613847, -0.10686638, -0.14147555,  0.8964462 ],
       [ 0.81227693, -0.38092692,  0.22995024, -0.37712564],
       [ 0.40613847,  0.72910447, -0.5208777 , -0.17928924]]), array([[ 9.8488578 ,  6.49821546, 10.55960012, 11.37187705],
       [ 0.        ,  5.98106979,  8.4234836 , -0.484346  ],
       [ 0.        ,  0.        ,  2.74586406,  3.27671222],
       [ 0.        ,  0.        ,  0.        ,  3.11592335]]))
QR-> matrices by python built-function

QRResult(Q=array([[-0.10153462,  0.558463  ,  0.80981107,  0.1483773 ],
       [-0.40613847, -0.10686638, -0.14147555,  0.8964462 ],
       [-0.81227693, -0.38092692,  0.22995024, -0.37712564],
       [-0.40613847,  0.72910447, -0.5208777 , -0.17928924]]), R=array([[ -9.8488578 ,  -6.49821546, -10.55960012, -11.37187705],
       [  0.        ,   5.98106979,   8.4234836 ,  -0.484346  ],
       [  0.        ,   0.        ,   2.745864

"\n\nThis QR decomposition are'nt unique because the only sign convention is the main difference in the Gram-Schmidt process\nThen the resultant should still give a when you do a matrix product of QR. \n\n"

(b) Using your function, write a `Python` program that determines the eigenvalues and eigenvectors of a real-valued symmetric matrix. Test it with the matrix above and test your results with the appropriate `numpy` function. 

In [4]:
#Question 2 Part B.) 
import numpy as np 
import pandas as pd 
import math
import matplotlib.pyplot as plt 
from scipy.linalg import lu 
import numpy.linalg as linalg
from sympy import init_printing, Matrix, pprint 



def QR_Eigen(C, max_iter = 1000): 
    '''
    Function takes in the result from the Gram-Schmidt function 
    and applies it, and iterates up to a certain limit where V = Q_1*Q_2*Q_3*...Q_n (eigenvector) 
    Also, it iterates C = R*Q, which can be put in similarity form. All of this converges to get eigenvalues and eigenvectors.

    Parameter
    ---------
    C: numpy.ndarray
       This is the input matrix of interest

    max_iter: int 
        This is an iterative limit to reach convergence 

    Returns
    -------
        numpy.ndarray
        Returns eigenvalues and eigenvectors in numpy array form. 
    '''

    n = C.shape[0]
    C_k = C.copy() 
    V = np.eye(n) 
    for k in range(max_iter): 
        Q,R = QR_Decomp(C_k)
        C_k = R@Q 
        V = V@Q 
    eignval = np.diag(C_k) 
    eignvect = V
    return eignval, eignvect

print("Function built, eigenvalues and eigenvectors of A: \n") 
print(QR_Eigen(A, max_iter = 1000))

#Python eigenfunction and eigenvalue built-in
w,v = linalg.eig(A)
print("\n Python built-in, eigenvalues and eigenvectors of A: \n") 
print("\n eigenvalues: ")
print(w) 
print("\n eigenvectors: ")
print(v) 





Function built, eigenvalues and eigenvectors of A: 

(array([21., -8., -3.,  1.]), array([[ 0.43151697, -0.38357064, -0.77459667, -0.25819889],
       [ 0.38357064,  0.43151697, -0.25819889,  0.77459667],
       [ 0.62330229,  0.52740963,  0.25819889, -0.51639778],
       [ 0.52740963, -0.62330229,  0.51639778,  0.25819889]]))

 Python built-in, eigenvalues and eigenvectors of A: 


 eigenvalues: 
[21. -8. -3.  1.]

 eigenvectors: 
[[-0.43151697 -0.38357064  0.77459667  0.25819889]
 [-0.38357064  0.43151697  0.25819889 -0.77459667]
 [-0.62330229  0.52740963 -0.25819889  0.51639778]
 [-0.52740963 -0.62330229 -0.51639778 -0.25819889]]


## Matrices and eigenvalue problems

Consider the matrix:
$$
A = 
\begin{pmatrix}
4 & -2 & 1 \\ 3 & 6 & -4 \\ 2 & 1 & 8
\end{pmatrix}
$$

(a) Evaluate the determinant of $A$ both analytically and computationally. Don't use any library implementation.

[Question 3a](https://drive.google.com/file/d/1wrHEC18hoNfgqO_02gVhCtrNX5rjE50n/view?usp=sharing)

In [5]:
#Question 3 Part a.) 
import numpy as np 
import pandas as pd 
import math
import matplotlib.pyplot as plt 
from scipy.linalg import lu 
import numpy.linalg as linalg
from sympy import init_printing, Matrix, pprint 


A = np.array([[4, -2, 1], [3, 6, -4], [2, 1, 8]], float) 


def QR_Decomp(B): 
    '''
    A Gram-Schmidt process is applied to the column vector of matrix B input. 
    Main equations used R_ij = Q^T*v; v = v - R_ij*Q; R_jj = 1/sqrt(n^2); and Q_ij = v/R_jj. 
    The final goal is to obtain the R and Q matrices in decomposed form from the input matrix. 
    
    Parameters
    ----------
    B : numpy.ndarray
       Input a real and square matrix. 

    Returns
    -------
    numpy.ndarray
        Both decomposed from B: Q and R matrices are output.
    '''
    m,n = B.shape
    Q = np.zeros((m,n))
    R = np.zeros((n,n))
    for j in range(n): 
        v = B[:,j].copy()
        for i in range(j):
            R[i,j] = np.dot(Q[:, i], v)
            v -= R[i,j]*Q[:, i]
        R[j,j] = linalg.norm(v)
        Q[:,j] = v/(R[j,j]) 
    return Q,R 

            

def Det(B): 
    '''
    Function that determines det(matrix) for square (nxn) matrices by QR decomposition 
    det(A) = det(Q)*det(R); det(Q) = +/- 1; det(R) = R_11*R_22*R_33*R_nn. 
    
    Parameters
    ----------
    B : numpy.ndarray
       Input a square matrix. 

    Returns
    -------
    float
        Determinant of A by QR decomposition. 
    '''
    Q, R = QR_Decomp(B) 
    # determinant of Q first 
    if linalg.det(Q) == -1:
        detQ = -1.0 
    else: 
        detQ = 1.0
    n = R.shape[0]
    #determinant of R 
    detR = 1.0
    for i in range(n): 
        detR *= R[i,i] 
    detB = detQ*detR 
    return detB 

print("det(A) = \n", Det(A))







det(A) = 
 263.0


(b) Verify both analytically and computationally that the inverse of $A$ is
\begin{equation}
A^{-1} = \frac{1}{263} \left( \begin{array}{ccc} 52 & 17 & 2 \\ 
                     -32 & 30 & 19 \\ -9 & -8 & 30 \end{array} \right)
\end{equation}
Don't use any library implementation.

[Question 3b](https://drive.google.com/file/d/1NnbomKBwdpllmXlM8W4piRd6anSfgLIO/view?usp=sharing)

In [6]:

#Question 3 Part b.) 
import numpy as np 
import pandas as pd 
import math
import matplotlib.pyplot as plt 
from scipy.linalg import lu 
import numpy.linalg as linalg
from sympy import init_printing, Matrix, pprint 

A = np.array([[4, -2, 1], [3, 6, -4], [2, 1, 8]], float) 



def Inverse(F):
    '''
    Function that computes the inverse of a square  matrix using 
    Gauss–Jordan elimination on the augmented matrix [F | I].

    Parameters
    ----------
    F : numpy.ndarray
        Input square matrix.

    Returns
    -------
    numpy.ndarray
        The inverse of F.
    '''
    n = F.shape[0]
    AI = np.hstack([F, np.eye(n)])

    # Forward elimination
    for i in range(n):
        for r in range(i + 1, n):
            m = AI[r, i]/AI[i,i]
            AI[r, :] -= m * AI[i, :]
    # Backward substitution
    for p in range(n - 1, -1, -1):
        for k in range(p - 1, -1, -1):
            m = AI[k, p]/AI[p,p]
            AI[k, :] -= m * AI[p, :]

    x = divisor(n, AI)
    inv = x[:, n:]
    return inv

print(Inverse(A))

    

[[ 0.19771863  0.06463878  0.00760456]
 [-0.121673    0.11406844  0.07224335]
 [-0.03422053 -0.03041825  0.11406844]]


(c) As a general procedure, applicable even if you do not know the analytic answer,
check the your inverse in both directions, i.e. verify both analytically and computationally that

\begin{equation}
AA^{-1} = A^{-1} A = {\bf 1}
\end{equation}

[Question 3 Part C.](https://drive.google.com/file/d/1ZgszUAG-aX8THKAxCxMwXfu2W_i4-o0f/view?usp=sharing)

In [7]:
#Question 3 Part c.) 
import numpy as np 
import pandas as pd 
import math
import matplotlib.pyplot as plt 
from scipy.linalg import lu 
import numpy.linalg as linalg
from sympy import init_printing, Matrix, pprint 

A = np.array([[4, -2, 1], [3, 6, -4], [2, 1, 8]], float) 

print("A*A^-1 = \n", np.dot(Inverse(A), A))
print("A^-1*A = \n", np.dot(A, Inverse(A)))


A*A^-1 = 
 [[ 1.00000000e+00 -1.04083409e-17  0.00000000e+00]
 [ 2.77555756e-17  1.00000000e+00  0.00000000e+00]
 [ 2.77555756e-17  0.00000000e+00  1.00000000e+00]]
A^-1*A = 
 [[ 1.00000000e+00 -2.08166817e-17  0.00000000e+00]
 [-2.77555756e-17  1.00000000e+00 -5.55111512e-17]
 [ 0.00000000e+00 -2.77555756e-17  1.00000000e+00]]


(d) Solve $Ax = B_1$ for 
\begin{equation}
B_1 = \begin{pmatrix} +4 \\ -10 \\ +22 \end{pmatrix} .
\end{equation} 
manually using Gauss elimination. Do not use any other method. Verify your results using a standard matrix solver _and_ `sympy`.

[Question 3 Part D.](https://drive.google.com/file/d/1R6RMUOWXXikndtfq35wtyW8aR4R8f3u6/view?usp=sharing)

In [8]:
#Question 3 Part d.) 
import numpy as np 
import pandas as pd 
import math
import matplotlib.pyplot as plt 
from scipy.linalg import lu 
import numpy.linalg as linalg
from sympy import init_printing, Matrix, pprint 


A = np.array([[4, -2, 1], [3, 6, -4], [2, 1, 8]], float) 
B_1 = np.array([4,-10,22],float)

print("x's = \n ", linalg.solve(A,B_1))


x's = 
  [ 0.31178707 -0.03802281  2.67680608]


In [9]:
#Question 3. Part d.) 

from sympy import Matrix

A = Matrix([[4, -2, 1],
            [3, 6, -4],
            [2, 1, 8]])
B1 = Matrix([4,-10,22])
AB1 = A.row_join(B1)
print("Solutions = \n", AB1.rref())





Solutions = 
 (Matrix([
[1, 0, 0,  82/263],
[0, 1, 0, -10/263],
[0, 0, 1, 704/263]]), (0, 1, 2))


Consider the matrix

\begin{equation}
A = 
\begin{pmatrix}
-2 & +2 & -3 \\ +2 & +1 & -6 \\ -1 & -2 & 0
\end{pmatrix}
\end{equation}

(a) Verify analytically and with `sympy` that the eigenvalues are $\lambda_1=5$, $\lambda_2=\lambda_3=-3$. 

[Question 4 Part A](https://drive.google.com/file/d/1YvjKvl6xWpJPy41bAPythZ3AdVZVHvh1/view?usp=sharing)

In [10]:
#Question 4. Part A.) 
from sympy import Matrix

A = Matrix([[-2,2,-3], [2,1,-6], [-1,-2,0]])
print("eigenvalues = \n", A.eigenvals())


eigenvalues = 
 {5: 1, -3: 2}


(b) Verify analytically and with `sympy` that the eigenvector for $\lambda_1=5$ is proportional to 

\begin{equation}
X_1 = \left( \begin{array}{c} -1 \\ -2 \\ +1  \end{array} \right).
\end{equation} 


[Question 4 Part B](https://drive.google.com/file/d/1Y08h3Dh_3LxOrxw-P1QlPDAo_ZEw14tF/view?usp=sharing)

In [11]:
#Question 4. Part B.) 
from sympy import Matrix

A = Matrix([[-2,2,-3], [2,1,-6], [-1,-2,0]])
print("eigenvalues = \n", A.eigenvects())




eigenvalues = 
 [(-3, 2, [Matrix([
[-2],
[ 1],
[ 0]]), Matrix([
[3],
[0],
[1]])]), (5, 1, [Matrix([
[-1],
[-2],
[ 1]])])]


(c) The eigenvalue $\lambda_{2,3} = -3$ corresponds to a degenerate root of the characteristic polynomials, which means that any linear combination of two given eigenvectors are also eigenvectors (associated with eigenvalues $\lambda_{2,3}$). Two linearly independent eigenvectors are

\begin{equation}
X_2 = \left( \begin{array}{c} -2 \\ +1 \\ 0  \end{array} \right) , \quad
X_3 = \left( \begin{array}{c} 3 \\ 0 \\ 1  \end{array} \right).
\end{equation}

In this case, it is not clear what your eigenproblem solver will return as eigenvectors. Find a relationship between your computed eigenvectors to the above given ones.

 **Question 4c**
 ----------------------------------------------------------------------------------------------------------
 relationship between computed eigenvectors and one's displayed above, if you augment both matrices, and want to see if their any matrix transformation between the eigenbasis visually, you can see the identity matrix $\textbf{I}$ is the relationship between them. 